# Credit Risk Analysis — SQL-First Assessment
**Role:** Data Analyst, Risk Strategy  
**Dataset:** [Credit Risk Dataset (Kaggle)](https://www.kaggle.com/datasets/laotse/credit-risk-dataset)  
**Engine:** SQLite (via Python `sqlite3`) — all core analysis done in pure SQL  
**Objective:** Identify ≤3 insights leadership should act on immediately.

---
## Table of Contents
1. [Setup & Data Cleaning](#setup)
2. [Portfolio Overview](#overview)
3. [Insight 1 — Loan Grade is the Dominant Default Predictor](#insight1)
4. [Insight 2 — DTI Cliff at 30%](#insight2)
5. [Insight 3 — Prior Default History & Pricing Adequacy](#insight3)
6. [Compound Risk Segment](#compound)
7. [Supplemental — Loan Intent](#intent)
8. [Priority & Next Question](#priority)
9. [Visualisations](#viz)

---
## 1. Setup & Data Cleaning <a id='setup'></a>

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

file_path = 'credit_risk_dataset.csv'   # update path as needed

df = pd.read_csv(file_path)
print(f"Raw rows loaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Raw rows loaded: 32,581
Columns: ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']


### 1a. Initial data inspection

In [2]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [3]:
df.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.000000,3.258100e+04,31686.000000,32581.000000,29465.000000,32581.000000,32581.000000,32581.000000
mean,27.734600,6.607485e+04,4.789686,9589.371106,11.011695,0.218164,0.170203,5.804211
std,6.348078,6.198312e+04,4.142630,6322.086646,3.240459,0.413006,0.106782,4.055001
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


### 1b. Data quality issues identified

> Note: Thresholds (age < 100, emp_length < 60) are conservative estimates. The filter removes clear data entry errors while preserving borderline-plausible records.

In [ ]:
#Step 1: Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {before - len(df)}")

#Step 2: Filter impossible ages 
before = len(df)
df = df[df['person_age'] < 100]
print(f"Rows removed (age >= 100): {before - len(df)}")

#Step 3: Filter impossible employment length 
before = len(df)
df = df[df['person_emp_length'] < 60]
print(f"Rows removed (emp_length >= 60): {before - len(df)}")

#Step 4: Drop rows with missing employment length 
before = len(df)
df = df.dropna(subset=['person_emp_length'])
print(f"Rows removed (emp_length null): {before - len(df)}")

#Step 5: Drop rows with missing interest rate 
before = len(df)
df = df.dropna(subset=['loan_int_rate'])
print(f"Rows removed (loan_int_rate null): {before - len(df)}")

Duplicates removed: 0
Rows removed (age >= 100): 0
Rows removed (emp_length >= 60): 0
Rows removed (emp_length null): 0
Rows removed (loan_int_rate null): 0


In [23]:
# Verify clean ranges
df.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,28495.000000,2.849500e+04,28495.000000,28495.000000,28495.000000,28495.000000,28495.000000,28495.000000
mean,27.723530,6.643047e+04,4.781751,9657.366205,11.045220,0.217126,0.169517,5.800316
std,6.177199,5.151374e+04,4.037958,6327.711290,3.230786,0.412296,0.106379,4.040800
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.948000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.600000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,8.000000e+04,7.000000,12500.000000,13.480000,0.000000,0.230000,8.000000
max,84.000000,2.039784e+06,41.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


In [25]:
con = sqlite3.connect(':memory:')
df.to_sql('loans', con, if_exists='replace', index=False)

def sql(query):
    return pd.read_sql(query, con)

print(f"{len(df):,} clean rows loaded into SQLite table: loans")

28,495 clean rows loaded into SQLite table: loans


> **Schema note:** `loan_status = 1` = default, `0` = no default.  
> `loan_percent_income` = loan amount ÷ annual income (DTI proxy).  
> `cb_person_default_on_file` = Y/N flag for prior default history.

---
## 2. Portfolio Overview <a id='overview'>

In [26]:
sql("""
SELECT
    COUNT(*)                                    AS total_loans,
    SUM(loan_status)                            AS total_defaults,
    ROUND(AVG(loan_status)*100, 2)              AS default_rate_pct,
    ROUND(AVG(loan_amnt), 0)                    AS avg_loan_amount,
    ROUND(AVG(loan_int_rate), 2)                AS avg_int_rate,
    COUNT(DISTINCT loan_grade)                  AS num_grades,
    COUNT(DISTINCT loan_intent)                 AS num_intents
FROM loans;
""")

,total_loans,total_defaults,default_rate_pct,avg_loan_amount,avg_int_rate,num_grades,num_intents
0,28495,6187,21.71,9657.0,11.05,7,6


---
## 3. Insight 1 — Loan Grade is the Dominant Default Predictor <a id='insight1'></a>

**Hypothesis:** The internal loan grade assigned at origination should correlate with actual default outcomes. If true, it is the single most actionable segmentation variable.

**Strategies:** Proposed strategies are hypothetical solution that might elevate choke points from segmentation.

### 3a. Default rate by grade

In [27]:
sql("""
SELECT
    loan_grade,
    COUNT(*)                                                        AS num_loans,
    SUM(loan_status)                                                AS num_defaults,
    ROUND(AVG(loan_status)*100, 2)                                  AS default_rate_pct,
    ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM loans), 1)           AS pct_of_portfolio,
    ROUND(AVG(loan_int_rate), 2)                                    AS avg_int_rate,
    ROUND(AVG(loan_amnt), 0)                                        AS avg_loan_amnt
FROM loans
GROUP BY loan_grade
ORDER BY loan_grade;
""")

,loan_grade,num_loans,num_defaults,default_rate_pct,pct_of_portfolio,avg_int_rate,avg_loan_amnt
0,A,9344,898,9.61,32.8,7.35,8580.0
1,B,9092,1448,15.93,31.9,11.00,10051.0
2,C,5680,1155,20.33,19.9,13.45,9305.0
3,D,3242,1920,59.22,11.4,15.35,10879.0
4,E,869,562,64.67,3.0,17.00,12934.0
5,F,209,146,69.86,0.7,18.60,15308.0
6,G,59,58,98.31,0.2,20.25,18142.0


### 3b. Risk tier aggregation: A–B vs C–D vs E–G

In [28]:
sql("""
SELECT
    CASE
        WHEN loan_grade IN ('E','F','G') THEN 'Grade E-G  (High Risk)'
        WHEN loan_grade IN ('A','B')     THEN 'Grade A-B  (Low Risk)'
        ELSE                                  'Grade C-D  (Medium Risk)'
    END                                                             AS risk_tier,
    COUNT(*)                                                        AS num_loans,
    ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM loans), 1)           AS pct_of_portfolio,
    ROUND(AVG(loan_status)*100, 2)                                  AS default_rate_pct,
    ROUND(AVG(loan_int_rate), 2)                                    AS avg_int_rate,
    ROUND(SUM(loan_amnt), 0)                                        AS total_exposure
FROM loans
GROUP BY 1
ORDER BY default_rate_pct DESC;
""")

,risk_tier,num_loans,pct_of_portfolio,default_rate_pct,avg_int_rate,total_exposure
0,Grade E-G (High Risk),1137,4.0,67.37,17.47,15509075.0
1,Grade C-D (Medium Risk),8922,31.3,34.47,14.14,88120675.0
2,Grade A-B (Low Risk),18436,64.7,12.73,9.15,171556900.0


### 🔍 Insight 1 Summary

| Dimension | Finding |
|---|---|
| **Stat** | Default rates span **9.61% (Grade A) → 98.31% (Grade G)** — a ~10× spread across the grading spectrum |
| **Grade D** | 11.4% of portfolio, 59.22% default rate, avg loan $10,879 — the largest high-risk segment by volume |
| **Grade E–G** | 4.0% of portfolio, 67.37% default rate, $15.5M total exposure — highest default concentration |
| **Pricing check** | Grade G charges avg 20.25% rate but defaults at 98.31% — the interest premium does not come close to covering expected losses |

- **Grade F/G:** Restrict originations through tighter underwriting criteria (e.g. minimum income thresholds) Apply maximum risk-adjusted pricing. Cap individual loan exposure. Monitor portfolio profitability quarterly, if loss rates remain above pricing breakeven, tighten criteria further.
- **Grade E:** Mandatory credit committee sign-off per application. Require guarantor or collateral. 
- **Grade D:** Enhanced monitoring cadence (monthly review). Flag for early intervention at first missed payment.

---
## 4. Insight 2 — DTI Cliff at 30% <a id='insight2'></a>

**Hypothesis:** `loan_percent_income` (loan amount ÷ annual income) is an affordability proxy. A threshold effect is expected — borrowers over-extending income should show disproportionately higher default rates.

### 4a. Default rate by DTI bucket

In [30]:
sql("""
SELECT
    CASE
        WHEN loan_percent_income <= 0.10 THEN '0-10%'
        WHEN loan_percent_income <= 0.20 THEN '10-20%'
        WHEN loan_percent_income <= 0.30 THEN '20-30%'
        WHEN loan_percent_income <= 0.40 THEN '30-40%'
        WHEN loan_percent_income <= 0.50 THEN '40-50%'
        ELSE                                  '>50%'
    END                                                   AS dti_bucket,
    COUNT(*)                                              AS num_loans,
    ROUND(AVG(loan_status)*100, 2)                        AS default_rate_pct,
    ROUND(AVG(loan_amnt), 0)                              AS avg_loan_amnt
FROM loans
GROUP BY 1
ORDER BY 1;
""")

,dti_bucket,num_loans,default_rate_pct,avg_loan_amnt
0,0-10%,9218,11.87,5296.0
1,10-20%,10569,14.87,9714.0
2,20-30%,5415,21.81,13158.0
3,30-40%,2334,69.24,15318.0
4,40-50%,743,74.29,17118.0
5,>50%,216,79.63,18426.0


### 4b. Binary split at the 30% threshold

In [31]:
sql("""
SELECT
    CASE WHEN loan_percent_income <= 0.30 THEN 'DTI <= 30%'
         ELSE                                  'DTI > 30%' END      AS dti_segment,
    COUNT(*)                                                         AS num_loans,
    ROUND(AVG(loan_status)*100, 2)                                   AS default_rate_pct,
    ROUND(
        AVG(loan_status)*100
        / (SELECT AVG(loan_status)*100 FROM loans WHERE loan_percent_income <= 0.30)
    , 2)                                                             AS lift_vs_below30
FROM loans
GROUP BY 1
ORDER BY 1 DESC;
""")

,dti_segment,num_loans,default_rate_pct,lift_vs_below30
0,DTI > 30%,3293,71.06,4.66
1,DTI <= 30%,25202,15.26,1.00


### 🔍 Insight 2 Summary

| Dimension | Finding |
|---|---|
| **Stat** | Default rate rises gradually from **11.87% (DTI 0–10%)** to **21.81% (DTI 20–30%)**, then jumps sharply to **69.24% at DTI 30–40%** and **79.63% above 50%** |
| **Cliff** | The 30% DTI threshold marks a near-5× jump in default probability —> from 15.26% below to 71.06% above |
| **Lift** | **4.66×** higher default probability above the 30% threshold |
| **Volume at risk** | 3,293 loans (11.6% of portfolio) sit above 30% DTI |

**Strategy:**
- **DTI > 30%:** Treat as a high-risk flag in all origination decisions. Do not approve without a compensating factor (e.g. Grade A–B, strong collateral, guarantor). 
- **DTI 25–30% (amber zone):** Track this cohort separately for early performance signals.
- **DTI ≤ 25%:** Standard approval process. No additional hurdles.

---
## 5. Insight 3 — Prior Default History: Is the Pricing Gap Sufficient? <a id='insight3'></a>

**Hypothesis:** `cb_person_default_on_file` (Y/N) is one of the strongest behavioural signals in credit. The bank charges a higher rate to Y-flag borrowers, but is the premium large enough to compensate for the actual default lift?

### 5a. Default rate and current pricing by prior-default flag

In [32]:
sql("""
SELECT
    cb_person_default_on_file                                            AS prior_default_flag,
    COUNT(*)                                                             AS num_loans,
    ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM loans), 1)                AS pct_of_portfolio,
    ROUND(AVG(loan_status)*100, 2)                                       AS default_rate_pct,
    ROUND(AVG(loan_int_rate), 2)                                         AS avg_int_rate
FROM loans
GROUP BY 1
ORDER BY 1;
""")

,prior_default_flag,num_loans,pct_of_portfolio,default_rate_pct,avg_int_rate
0,N,23406,82.1,18.21,10.29
1,Y,5089,17.9,37.83,14.51


### 5b. Quantify the default lift and the rate gap

In [34]:
sql("""
SELECT
    -- How much more likely are Y-flag borrowers to default?
    ROUND(
        (SELECT AVG(loan_status) FROM loans WHERE cb_person_default_on_file = 'Y')
       /(SELECT AVG(loan_status) FROM loans WHERE cb_person_default_on_file = 'N')
    , 2)                                    AS default_rate_lift,

    -- How much extra rate are they charged today?
    ROUND(
        (SELECT AVG(loan_int_rate) FROM loans WHERE cb_person_default_on_file = 'Y')
       -(SELECT AVG(loan_int_rate) FROM loans WHERE cb_person_default_on_file = 'N')
    , 2)                                    AS current_rate_premium_pct
""")

,default_rate_lift,current_rate_premium_pct
0,2.08,4.22


### 🔍 Insight 3 Summary

| Dimension | Finding |
|---|---|
| **Default rate** | Y-flag: **37.83%** vs N-flag: **18.21%** —> a **2.08× lift** |
| **Current rate premium** | Y-flag borrowers pay **4.22% more** on average (14.51% vs 10.29%) |
| **Why it matters** | Y-flag borrowers contribute disproportionate expected losses that the current pricing does not recover |

**Strategy:**
- **Increase rate surcharge for Y-flag borrowers:** The current 4.22% premium is insufficient. 
- **Reduce loan exposure:** Cap maximum loan amount for Y-flag borrowers to reduce absolute expected loss per loan.
- **Require additional guarantor for Y-flag + Grade D+:** The combination of prior default history and lower grade is the highest-risk pairing, a guarantor or collateral requirement provides a structural loss buffer.

---
## 6. Compound Risk Segment <a id='compound'></a>

Combining Insights 1 and 2 to identify the **highest-risk pocket** in the portfolio.

In [35]:
sql("""
SELECT
    CASE
        WHEN loan_grade IN ('A','B','C') AND loan_percent_income <= 0.30
            THEN '1. Grade A-C  | DTI<=30%  (Low Risk)'
        WHEN loan_grade IN ('A','B','C') AND loan_percent_income  > 0.30
            THEN '2. Grade A-C  | DTI>30%   (Medium Risk)'
        WHEN loan_grade IN ('D','E','F','G') AND loan_percent_income <= 0.30
            THEN '3. Grade D+   | DTI<=30%  (Medium-High)'
        WHEN loan_grade IN ('D','E','F','G') AND loan_percent_income  > 0.30
            THEN '4. Grade D+   | DTI>30%   *** DANGER ZONE ***'
    END                                                      AS risk_segment,
    COUNT(*)                                                  AS num_loans,
    ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM loans), 1)    AS pct_of_portfolio,
    ROUND(AVG(loan_status)*100, 2)                           AS default_rate_pct,
    ROUND(SUM(loan_amnt), 0)                                 AS total_exposure,
    ROUND(SUM(loan_amnt * loan_status), 0)                   AS expected_loss
FROM loans
GROUP BY 1
ORDER BY 1;
""")

,risk_segment,num_loans,pct_of_portfolio,default_rate_pct,total_exposure,expected_loss
0,1. Grade A-C | DTI<=30% (Low Risk),21538,75.6,8.18,184796350.0,12471450.0
1,2. Grade A-C | DTI>30% (Medium Risk),2578,9.0,67.46,39611750.0,24917450.0
2,3. Grade D+ | DTI<=30% (Medium-High),3664,12.9,56.91,37940350.0,20297075.0
3,4. Grade D+ | DTI>30% *** DANGER ZONE ***,715,2.5,84.06,12838200.0,10536075.0


### Compound Segment Summary

| Segment | Loans | Portfolio % | Default Rate | Total Exposure | Expected Loss |
|---|---|---|---|---|---|
| Grade A–C \| DTI ≤30% (Low Risk) | 21,538 | 75.6% | 8.18% | $184.8M | $12.5M |
| Grade A–C \| DTI >30% (Medium Risk) | 2,578 | 9.0% | 67.46% | $39.6M | $24.9M |
| Grade D+ \| DTI ≤30% (Medium-High) | 3,664 | 12.9% | 56.91% | $37.9M | $20.3M |
| **Grade D+ \| DTI >30% (DANGER ZONE)** | **715** | **2.5%** | **84.06%** | **$12.8M** | **$10.5M** |

> **Danger Zone:** 715 loans representing just 2.5% of the portfolio drive $10.5M in expected losses at an 84% default rate. This is the single most urgent intervention target.

> **Segment 2 note:** Grade A–C borrowers above 30% DTI show a 67.46% default rate, nearly as alarming as the Danger Zone and involving far more loans (2,578). DTI is a dominant risk driver even for well-graded borrowers.

**Strategy:**
- **Danger Zone (D+ | DTI >30%):** Immediate review of all active loans. For new originations: require credit committee approval, mandatory collateral, guarantor, and restrict exposure to ≤50% of standard loan limit for that grade. Apply Insights 1 and 3 strategies simultaneously.
- **Segment 2 (A–C | DTI >30%):** Do not rely on good grade alone. Apply the 30% DTI ceiling policy from Insight 2, even A–B borrowers above this threshold carry a 60–67% default rate in this data.
- **Segment 3 (D+ | DTI ≤30%):** Tighter underwriting per Insight 1 strategy. Monitor early payment behaviour as a leading indicator.

### 6b. Grade × DTI interaction matrix

In [36]:
sql("""
SELECT
    loan_grade,
    ROUND(AVG(CASE WHEN loan_percent_income <= 0.30 THEN loan_status END)*100, 1)
        AS default_rate_dti_low,
    ROUND(AVG(CASE WHEN loan_percent_income  > 0.30 THEN loan_status END)*100, 1)
        AS default_rate_dti_high,
    COUNT(CASE WHEN loan_percent_income  > 0.30 THEN 1 END)
        AS n_loans_dti_high
FROM loans
GROUP BY loan_grade
ORDER BY loan_grade;
""")

,loan_grade,default_rate_dti_low,default_rate_dti_high,n_loans_dti_high
0,A,4.9,60.9,780
1,B,8.4,67.0,1161
2,C,13.3,76.3,637
3,D,55.2,82.3,486
4,E,59.2,87.9,165
5,F,66.5,82.2,45
6,G,97.5,100.0,19


> **Key interaction:** Even Grade A borrowers jump from **4.9% → 60.9%** default rate when DTI exceeds 30%. Grade G at DTI >30% hits **100%** default. DTI is not just an additive risk factor —> it is multiplicative across all grades.

---
## 7. Supplemental — Default Rate by Loan Intent <a id='intent'></a>

In [37]:
sql("""
SELECT
    loan_intent,
    COUNT(*)                                             AS num_loans,
    ROUND(AVG(loan_status)*100, 2)                       AS default_rate_pct,
    ROUND(AVG(loan_amnt), 0)                             AS avg_loan_amnt
FROM loans
GROUP BY loan_intent
ORDER BY default_rate_pct DESC;
""")

,loan_intent,num_loans,default_rate_pct,avg_loan_amnt
0,DEBTCONSOLIDATION,4547,28.46,9674.0
1,MEDICAL,5269,26.91,9353.0
2,HOMEIMPROVEMENT,3187,25.73,10409.0
3,PERSONAL,4857,19.77,9667.0
4,EDUCATION,5668,17.06,9512.0
5,VENTURE,4967,14.66,9639.0


| Loan Intent | Default Rate | Note |
|---|---|---|
| Debt Consolidation | **28.46%** | Highest — borrowers likely already under financial stress |
| Medical | 26.91% | Unplanned expense, income disruption risk |
| Home Improvement | 25.73% | Elevated but more asset-backed in nature |
| Personal | 19.77% | Near portfolio average |
| Education | 17.06% | Lower risk, likely stable future income expectation |
| Venture | **14.66%** | Lowest — possibly self-selecting for financially confident borrowers |

Debt Consolidation and Medical loans show structurally higher default rates, suggesting intent-specific underwriting overlays may be warranted —> particularly when combined with lower grades or high DTI.

---
## 8. Priority & Next Question <a id='priority'></a>

### If leadership can only act on one insight: **Insight 2: DTI Cliff at 30%**

---
- **Sharpest single rule:** One hard number at origination. DTI >30% triggers a 4.66× default lift.
- **Overrides grade:** A Grade A borrower at DTI >30% defaults at 60.9%, worse than a Grade D borrower below 30% DTI (55.2%). Good grade gives false confidence if DTI is unchecked. DTI is the dominant risk driver regardless of where a borrower sits on the grading scale.
- **Broader portfolio impact:** The 30% DTI rule covers 3,293 loans (11.6% of the portfolio) across all grades. Grade F/G tightening by contrast only touches 268 loans (0.9%). Same policy effort, 13× the coverage.
- **Immediately deployable:** DTI is a calculated field at origination, income and loan amount are always known. No system changes, no new data, no model retraining required.
- **Grade becomes the triage tool within DTI, not the primary gate:** Once you enforce the DTI ceiling, loan grade tells you how aggressively to price and monitor the borderline cases in the 25–30% amber zone.

---

### One additional question for further investigation

> **Does loan intent interact with grade and DTI to create distinct risk micro-segments?**

Debt consolidation loans default at 28.46%, the highest of any intent category. A three-way interaction: `grade × loan_percent_income × loan_intent` could reveal whether debt consolidation or medical loans warrant product-specific underwriting overlays, independent of what the borrower's grade already captures. This would be the next SQL segmentation.

---